<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_landmark_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial - GarmentIQ Landmark Detection

Landmark detection locates the key points of a garment, such as shoulders, sleeve ends,
and hems. These points are what GarmentIQ measures between, so accurate landmarks lead
directly to accurate measurements.

This tutorial shows how to load the pretrained HRNet model, detect the predefined
landmarks for a known garment class, read the confidence scores, and plot the results on
the original image.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Detect landmarks](#detect)
3. [Understand the output](#output)

<a name="prerequisites"></a>
## Prerequisites

Install the package and download the test image and the pretrained weights. On Colab you
can keep this section collapsed.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import torch

import garmentiq as giq
from garmentiq.landmark.detection.model_definition import PoseHighResolutionNet
from garmentiq.garment_classes import garment_classes

# GarmentIQ never grabs an accelerator on its own: every model loader and every
# inference function takes a `device` argument that defaults to "cpu". Pass it
# explicitly to use a GPU ("cuda") or Apple Silicon ("mps").
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

In [ ]:
# @title Download the test image and the pretrained model

!mkdir -p ./test_image
!wget -q -O ./test_image/cloth_3.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_3.jpg

!mkdir -p ./models
!wget -q -O ./models/hrnet.pth \
    https://huggingface.co/lygitdata/garmentiq/resolve/main/hrnet.pth

print("Downloads finished.")

<a name="detect"></a>
## Detect landmarks

Landmark detection needs to know the garment class, because each class has its own set of
landmarks. `garment_classes` holds the predefined ones. The test image is a vest dress.

In [ ]:
print("Available garment classes:")
for name in garment_classes:
    print(" -", name)

giq.landmark.plot(image_path="./test_image/cloth_3.jpg", figsize=(3, 3))

In [ ]:
# Step 1: load the model. Note that `model_class` is an instance here, not a class.
HRNet = giq.landmark.detection.load_model(
    model_path="./models/hrnet.pth",
    model_class=PoseHighResolutionNet(),
    device=device,
)

In [ ]:
# Step 2: detect
coords, maxvals, detection_dict = giq.landmark.detect(
    class_name="vest dress",
    class_dict=garment_classes,
    image_path="./test_image/cloth_3.jpg",
    model=HRNet,
    scale_std=200.0,
    resize_dim=[288, 384],
    normalize_mean=[0.485, 0.456, 0.406],
    normalize_std=[0.229, 0.224, 0.225],
    device=device,
)

giq.landmark.plot(
    image_path="./test_image/cloth_3.jpg",
    coordinate=coords,
    figsize=(3, 3),
    color="green",
)

<a name="output"></a>
## Understand the output

`detect` returns three things:

| Value | Meaning |
|---|---|
| `coords` | landmark coordinates in pixels, shaped `(1, n_landmarks, 2)` |
| `maxvals` | the model's confidence for each landmark, from 0 to 1 |
| `detection_dict` | the full per-landmark record, keyed by landmark ID |

`detection_dict` is what the refinement, derivation, and measurement stages consume.

In [ ]:
print("coords shape :", coords.shape)
print("maxvals shape:", maxvals.shape)

for i, (xy, conf) in enumerate(zip(coords[0], maxvals[0])):
    print(f"landmark {i + 1:>2}: x={xy[0]:7.1f}  y={xy[1]:7.1f}  confidence={conf.item():.3f}")

Detected landmarks can sit slightly off the true garment edge, and some
useful points are not predicted at all. Both problems are solved in the
[landmark refinement and derivation tutorial](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_landmark_refinement_and_derivation.ipynb).